In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# page 135
from langchain_community.document_loaders import PDFPlumberLoader
import pdfplumber

loader = PDFPlumberLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")

docs = loader.load()
docs = docs[3:-1]
len(docs)

19

In [3]:
# page 136
docs[0].metadata

{'source': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'page': 3,
 'total_pages': 23,
 'Author': 'dj',
 'Creator': 'Hwp 2018 10.0.0.13462',
 'Producer': 'Hancom PDF 1.3.0.542',
 'CreationDate': "D:20231208132838+09'00'",
 'ModDate': "D:20231208132838+09'00'",
 'PDFVersion': '1.4'}

In [4]:
for doc in docs:
    doc.metadata["filename"] = doc.metadata["source"]

In [5]:
from ragas.testset import TestsetGenerator

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 데이터셋 생성 모델
generator_llm = ChatOpenAI(model="gpt-4o-mini")
# 임베딩 모델
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Ragas 0.4.x에서는 KeyphraseExtractor와 critic_llm을 직접 구성하지 않습니다.
generator = TestsetGenerator.from_langchain(
    llm=generator_llm,
    embedding_model=embeddings,
)

d:\skc0902\rag_two_branch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# page 138
from ragas.testset.transforms                       import SummaryExtractor, EmbeddingExtractor
from ragas.testset.transforms.extractors            import NERExtractor
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer

transforms = [                                                      # 페이지를 분할하지 않고 각 DOCUMENT에 필요한 정보를 추출
    
    SummaryExtractor(llm=generator.llm),                            # LLM으로 문서 내용을 요약

    EmbeddingExtractor(                                             # 지정한 텍스트 속성을 숫자 벡터로 변환
        embedding_model    =generator.embedding_model,
        property_name      ="summary_embedding",
        embed_property_name="summary",
    ),
    NERExtractor(llm=generator.llm),                                # 개체명 인식(Named Entity Recognition)으로 사람·기관·장소 등 주요 개체를 추출
]

query_distribution = [                                              # 한 페이지의 개체 정보를 바탕으로 질문 생성
    (SingleHopSpecificQuerySynthesizer(llm=generator.llm), 1.0),    # 하나의 문맥에서 답할 수 있는 구체적인 질문과 참조 정답을 생성
]

In [7]:
# 테스트셋 생성됨.
testset = generator.generate_with_langchain_docs(
    documents           =docs,                  # PDF 문서에서 추출한 페이지를 DOCUMENT로 변환하여 전달

    testset_size        =10,                    # 생성할 테스트셋의 질문 수

    transforms          =transforms,            # 각 DOCUMENT에 적용할 변환기(요약, 임베딩, 개체명 인식)

    query_distribution  =query_distribution,    # 한 페이지의 개체 정보를 바탕으로 질문 생성
    
    raise_exceptions    =True,                  # 질문 생성 중 오류 발생 시 예외를 발생시키도록 설정
)

Generating Samples: 100%|██████████| 10/10 [00:04<00:00,  2.28it/s]


In [8]:
# page 139
test_df = testset.to_pandas()

if test_df.empty:
    raise RuntimeError(
        "샘플이 0개입니다. 개체 추출 결과와 페르소나 매칭을 확인하세요."
    )

print(f"생성된 샘플 수: {len(test_df)}")
#test_df.head()
# print(test_df)
test_df

생성된 샘플 수: 10


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What is the significance of The White House's ...,"[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국, 안전하고...","The White House's recent executive order, sign...",AI Ethics Compliance Officer,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
1,G7이 첨단 AI 시스템을 위한 국제 행동강령(International Code o...,"[SPRi AI Brief |\n2023-12월호\nG7, 히로시마 AI 프로세스를...",G7은 2023년 10월 30일 '히로시마 AI 프로세스'를 통해 AI 기업을 대상...,AI Policy Analyst,MISSPELLED,LONG,single_hop_specific_query_synthesizer
2,영국 AI 안전성 정상회의에서 뭐가 발표되었나요?,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n영국 AI 안전...,영국 AI 안전성 정상회의에 참가한 28개국들은 AI 안전 보장을 위한 협력 방안을...,AI Ethics Compliance Officer,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
3,미드저니 저작권 소송 왜 기각됐나?,"[SPRi AI Brief |\n2023-12월호\n미국 법원, 예술가들이 생성 A...",미드저니에 대한 저작권 소송은 미국 캘리포니아 북부지방법원에서 기각됐다. 기각 이유...,AI Policy Advisor,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
4,저작권청은 뭐하는 곳이야?,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국 연방거래위...,"저작권청은 생성 AI와 관련된 저작권법과 정책 이슈를 조사하고 있으며, 폭넓은 의견...",AI Policy Advisor,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
5,How has Euractiv reported on the challenges fa...,"[SPRi AI Brief |\n2023-12월호\nEU AI 법 3자 협상, 기반...",Euractiv has reported that the negotiations fo...,AI Policy Advisor,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
6,오픈AI는 뭐하는 곳이야?,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국 프런티어 ...,"오픈AI는 구글, 앤스로픽, 마이크로소프트와 함께 프런티어 모델 포럼에 참여하여 A...",AI Policy Advisor,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
7,What role did MIT play in the launch of the Da...,"[SPRi AI Brief |\n2023-12월호\n코히어, 데이터 투명성 확보를 ...",MIT was one of the 12 institutions that collab...,AI Policy Advisor,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
8,What advancements were announced by Alibaba Cl...,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n알리바바 클라우...,During the technology conference on October 31...,AI Ethics Compliance Officer,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
9,How does the Samsung Gauss compare to the Goog...,"[SPRi AI Brief |\n2023-12월호\n삼성전자, 자체 개발 생성 AI...",The Samsung Gauss is expected to compete with ...,AI Policy Advisor,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer


In [9]:
test_df.to_csv("../data/ragas_synthetic_dataset.csv", index=False, encoding="utf-8-sig")
print("저장완료")

저장완료
